In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "충청남도_천안시_착한가격업소": "충청남도_천안시_착한가격업소_20260326.csv"
    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path, encoding='cp949')
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(
    supabase_db_url,
    include_tables=["충청남도 천안시_착한가격업소_20260326"]
)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(
    supabase_db_url,
    include_tables=["충청남도 천안시_착한가격업소_20260326"]
)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"[{table}] 테이블 샘플:")
    try:
        result = db.run(f'SELECT * FROM "{table}" LIMIT 3;')
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = 'SELECT "업소명", "업종", "소재지주소" FROM "충청남도 천안시_착한가격업소_20260326" WHERE "업종" = \'한식\' LIMIT 5;'

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = 'SELECT "업종", COUNT(*) FROM "충청남도 천안시_착한가격업소_20260326" GROUP BY "업종";'

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    당신은 SQL 전문가입니다.
    사용자의 질문을 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")



def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 [YOUR_DOMAIN] 데이터 분석 전문가입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문에 자연스럽게 답변하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "YOUR_QUESTION_HERE"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "천안시 착한가격 업소는 총 몇개인가요?",
    "천안시 착한가격 업소 중 한식 업종은 몇개인가요?",
    "천안시 착한가격 업소 중 중식 업종의 상호명과 주소를 알려주세요.",

]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")

        

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.
📋 충청남도_천안시_착한가격업소 테이블

행 수: 124
컬럼: ['연번', '지정번호', '업소명', '업종', '대표자', '읍면동', '행정동', '소재지주소', '데이터기준일자']

첫 5개 행:
   연번         지정번호        업소명    업종  대표자  읍면동   행정동  \
0   1  천안-2011-005     선비숯불갈비    한식  홍대의  신부동   신안동   
1   2  천안-2012-020     포인트미용실  이미용업  원용백  대흥동   중앙동   
2   3  천안-2012-021      스타미용실  이미용업  신필자  대흥동   중앙동   
3   4  천안-2012-025  챠밍헤어클럽미용실  이미용업  이일례  원성동  원성2동   
4   5  천안-2012-029      선경세탁소   세탁업  홍성찬  봉명동   봉명동   

                       소재지주소     데이터기준일자  
0      천안시 동남구 터미널3길 21(신부동)  2026-03-26  
1  천안시 동남구 공설시장2길 3, 3층(대흥동)  2026-03-26  
2       천안시 동남구 대흥로 271(대흥동)  2026-03-26  
3       천안시 동남구 고재4길 55(원성동)  2026-03-26  
4       천안시 동남구 양지4길 15(봉명동)  2026-03-26  

데이터 타입:
연번         int64
지정번호         str
업소명          str
업종           str
대표자          str
읍면동          str
행정동          str
소재지주소        str
데이터기준일자      str
dtype: object



✓ 총 1개의 테이블 로드 완료


C:\Users\kyzky\AppData\Local\Temp\ipykernel_9796\1042405310.py:58: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['충청남도 천안시_착한가격업소_20260326']
=== 데이터베이스 스키마 ===

CREATE TABLE "충청남도 천안시_착한가격업소_20260326" (
	"연번" BIGINT, 
	"지정번호" TEXT, 
	"업소명" TEXT, 
	"업종" TEXT, 
	"대표자" TEXT, 
	"읍면동" TEXT, 
	"행정동" TEXT, 
	"소재지주소" TEXT, 
	"데이터기준일자" TEXT
)

/*
3 rows from 충청남도 천안시_착한가격업소_20260326 table:
연번	지정번호	업소명	업종	대표자	읍면동	행정동	소재지주소	데이터기준일자
1	천안-2011-005	선비숯불갈비	한식	홍대의	신부동	신안동	천안시 동남구 터미널3길 21(신부동)	2026-03-26
2	천안-2012-020	포인트미용실	이미용업	원용백	대흥동	중앙동	천안시 동남구 공설시장2길 3, 3층(대흥동)	2026-03-26
3	천안-2012-021	스타미용실	이미용업	신필자	대흥동	중앙동	천안시 동남구 대흥로 271(대흥동)	2026-03-26
*/


[충청남도 천안시_착한가격업소_20260326] 테이블 샘플:
[(1, '천안-2011-005', '선비숯불갈비', '한식', '홍대의', '신부동', '신안동', '천안시 동남구 터미널3길 21(신부동)', '2026-03-26'), (2, '천안-2012-020', '포인트미용실', '이미용업', '원용백', '대흥동', '중앙동', '천안시 동남구 공설시장2길 3, 3층(대흥동)', '2026-03-26'), (3, '천안-2012-021', '스타미용실', '이미용업', '신필자', '대흥동', '중앙동', '천안시 동남구 대흥로 271(대흥동)', '2026-03-26')]

실행 쿼리:
SELECT "업소명", "업종", "소재지주소" FROM "충청남도 천안시_착한가격업소_20260326" WHERE "업종"

천안시의 착한가격업소가 총 **124개** 확인됩니다.  
업소 유형은 주로 **한식**과 **이미용업**이 많고, 그 외에도 **중식, 일식, 세탁업, 숙박업, 기타요식업, 기타비요식업, 미용업, 양식** 등이 포함되어 있습니다.

원하시면 제가 이 결과를 바탕으로 다음과 같이도 정리해드릴 수 있어요:
- **업종별 개수**
- **행정동별 개수**
- **상호명 목록**
- **서북구/동남구별 분포**

질문을 구체적으로 주시면 바로 분석해서 답변드리겠습니다.


질문: 천안시 착한가격 업소는 총 몇개인가요?

[1] SQL 생성 중...
    SELECT COUNT(*) AS "총개수"
FROM "충청남도 천안시_착한가격업소_20260326";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소는 **총 124개**입니다.


질문: 천안시 착한가격 업소 중 한식 업종은 몇개인가요?

[1] SQL 생성 중...
    SELECT COUNT(*) AS 한식_업소수
FROM "충청남도 천안시_착한가격업소_20260326"
WHERE "업종" = '한식';

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소 중 **한식 업종은 60개**입니다.


질문: 천안시 착한가격 업소 중 중식 업종의 상호명과 주소를 알려주세요.

[1] SQL 생성 중...
    SELECT "업소명", "소재지주소"
FROM "충청남도 천안시_착한가격업소_20260326"
WHERE "업종" = '중식';

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소 중 **중식 업종**의 상호명과 주소는 다음과 같습니다.

1. **청룡각** — 천안시 서북구 충무로 143-8(쌍용동)  
2. **북경중화요리** — 천안시 동남구 신부1길 6(신부동)  
3. **명윤** — 천안시 서북구 직산읍 4산단로 241, 1동  
4. **강짬뽕** — 천안시 동남구 신부12길 12, 1층(신부동)  
5. **성환반점** — 천안시 서북구 성환읍 성진로 27  
6. **홍콩앤홍교** — 천안시 서북구 한들1로 145(백석동)

원하시면 제가 **동별로 정리**해서도 보여드릴게요.